# Fase 0 — Herramientas de calibración de `sigma_0`

Implementa la Fase 0 de `PLAN_CALIBRACION.md` (§8): scripts de caracterización
experimental (0.1) y de extracción de resultados de simulación (0.2). **No
lanza ninguna simulación nueva** — solo lee ficheros ya existentes.

La validación de 0.2 (§8, Fase 0.3: reproducir `14.333 / 10.750 / 8.590 Ω` de
`Results_tamano_espacio/`, §5 del plan) ya se hizo al escribir este notebook —
`extraer_resultado_simulacion` reprodujo la tabla con diff < 0.003% — y no se
mantiene como celda aquí; el hallazgo que dejó (hace falta `Clean_state_matrix`,
ver nota más abajo) sí queda documentado.

Lee `PLAN_CALIBRACION.md` y `CLAUDE.md` antes de tocar esto. En particular:
- El objetivo es `R_0 = 41.14 \Omega` (punto frío, rama de bajada = SP_set, del
  ciclo experimental `Cycle_p_1000.txt`).
- La resistencia fría de una simulación se lee en el **límite T → T_0** (final de
  SP_set, §4 del plan: ahí toda celda de filamento está exactamente a 300.00 K),
  no reconstruida a mano con la fórmula cerrada salvo como cross-check.

In [1]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import numpy as np
import pandas as pd

from RRAM import CurrentSolver

ruta_raiz = Path.cwd()
print("Ruta raiz del proyecto:", ruta_raiz)

Ruta raiz del proyecto: /Users/antonio_lopez_torres/Documents/GitHub/RRAM_Simulation


## 0.1 — Caracterización experimental

Entrada: un fichero `Cycle_p_*.txt` (`[V, I, ?]`, **sin cabecera** — ojo, parte
del código legacy usa `skiprows=1`, aquí NO se hace). Salida: `R_0` (frío),
`R_hot` (caliente), su cociente, y el ajuste global de referencia.

Reglas fijadas en el plan (§8, Fase 0.1):
- Las ramas se separan por el índice del máximo de V (`argmax`), no por una
  hipótesis sobre el número de filas.
- El ajuste es lineal **forzado por el origen**: `G = sum(V*I) / sum(V**2)`,
  `R = 1/G`. No es una regresión con término independiente.
- Ventanas: frío `0 < V <= 0.25 V`, caliente `V >= 0.90 V`, ambas sobre la
  **rama de bajada** (que en SP_set es la que reproduce el filamento ya
  formado, ver §3 del plan).

In [2]:
def leer_ciclo_experimental(path):
    """Lee un fichero Cycle_p_*.txt: columnas [V, I, ?], sin cabecera."""
    datos = np.loadtxt(path)
    V, I = datos[:, 0], datos[:, 1]
    return V, I


def separar_ramas(V, I):
    """Separa subida (0 -> V_max) y bajada (V_max -> 0) por el indice del maximo de V.

    El indice del maximo se incluye en ambas ramas (es el punto de union).
    """
    idx_max = int(np.argmax(V))
    subida = (V[: idx_max + 1], I[: idx_max + 1])
    bajada = (V[idx_max:], I[idx_max:])
    return subida, bajada


def ajuste_por_origen(V, I, mask=None):
    """Ajuste lineal I = G*V forzado por el origen. Devuelve (R, R2, n_puntos).

    G = sum(V*I) / sum(V**2)  ->  R = 1/G. R2 calculado respecto a la media de I,
    solo como referencia de calidad de ajuste (no es el criterio de la ventana).
    """
    if mask is not None:
        V, I = V[mask], I[mask]
    if len(V) == 0:
        raise ValueError("Ventana vacia: ningun punto cumple la condicion de V.")
    G = np.sum(V * I) / np.sum(V**2)
    R = 1.0 / G
    I_pred = G * V
    ss_res = np.sum((I - I_pred) ** 2)
    ss_tot = np.sum((I - np.mean(I)) ** 2)
    R2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return R, R2, len(V)


def caracterizar_ciclo_experimental(path, v_frio=0.25, v_caliente=0.90):
    """Aplica el procedimiento completo de 0.1 a un fichero Cycle_p_*.txt.

    Devuelve un dict con R0 (frio), R_hot (caliente), su cociente, y el ajuste
    global de referencia (NO usar este ultimo como objetivo de sigma_0, ver plan).
    """
    V, I = leer_ciclo_experimental(path)
    _, (V_bajada, I_bajada) = separar_ramas(V, I)

    mask_frio = (V_bajada > 0) & (V_bajada <= v_frio)
    mask_caliente = V_bajada >= v_caliente

    R0, r2_frio, n_frio = ajuste_por_origen(V_bajada, I_bajada, mask_frio)
    R_hot, r2_hot, n_hot = ajuste_por_origen(V_bajada, I_bajada, mask_caliente)
    R_global, r2_global, n_global = ajuste_por_origen(V_bajada, I_bajada)

    return {
        "fichero": Path(path).name,
        "R0_frio": R0,
        "n_puntos_frio": n_frio,
        "R_hot": R_hot,
        "n_puntos_hot": n_hot,
        "ratio_hot_frio": R_hot / R0,
        "R_global": R_global,
        "R2_global": r2_global,
        "n_puntos_global": n_global,
    }

In [3]:
# Aplicado al ciclo 1000 (unico dentro de alcance, ver PLAN_CALIBRACION.md \u00a73)
ciclo_1000 = ruta_raiz / "Datos_Experimentales" / "Ciclos_Experimentales" / "Cycle_p_1000.txt"
caracterizacion_1000 = caracterizar_ciclo_experimental(ciclo_1000)
caracterizacion_1000

{'fichero': 'Cycle_p_1000.txt',
 'R0_frio': np.float64(41.14452986081941),
 'n_puntos_frio': 25,
 'R_hot': np.float64(42.130032224667374),
 'n_puntos_hot': 21,
 'ratio_hot_frio': np.float64(1.0239522086455148),
 'R_global': np.float64(41.84682809703443),
 'R2_global': np.float64(0.9997223568032825),
 'n_puntos_global': 111}

In [4]:
# Validacion cruzada contra la tabla del plan (\u00a73):
# R0 = 41.14 \u03a9, R_hot = 42.13 \u03a9, ratio = 1.024, R_global = 41.85 \u03a9 (R2 = 0.99972)
esperado = {"R0_frio": 41.14, "R_hot": 42.13, "ratio_hot_frio": 1.024, "R_global": 41.85}
for clave, valor_esperado in esperado.items():
    valor_obtenido = caracterizacion_1000[clave]
    diff_pct = 100 * abs(valor_obtenido - valor_esperado) / valor_esperado
    print(f"{clave:16s}  obtenido={valor_obtenido:10.4f}  esperado(plan)={valor_esperado:8.4f}  diff={diff_pct:.3f}%")
    assert diff_pct < 0.1, f"{clave} no reproduce el valor documentado en el plan"
print("\nOK: 0.1 reproduce la tabla del \u00a73 del plan.")

R0_frio           obtenido=   41.1445  esperado(plan)= 41.1400  diff=0.011%
R_hot             obtenido=   42.1300  esperado(plan)= 42.1300  diff=0.000%
ratio_hot_frio    obtenido=    1.0240  esperado(plan)=  1.0240  diff=0.005%
R_global          obtenido=   41.8468  esperado(plan)= 41.8500  diff=0.008%

OK: 0.1 reproduce la tabla del §3 del plan.


## 0.2 — Extracción de resultados de simulación

Entrada: un directorio `Results_*/simulation_{N}/`. Salida: una fila con
`(sigma_0_usada, w_total, R_frio_sim)` por simulación.

Reglas fijadas en el plan (§8, Fase 0.2 y §9 "Trampas conocidas"):
- `sigma_0` y `cf_ranges` se leen de `sim_metadata_{N}.json` (`ctes_dict` /
  top-level). **Nunca hardcodear.**
- La columna 0 de `datos_sim` NO es el paso (recorre 10–19 en toda la rama);
  hay que indexar por posición, no por valor.
- `w_total` real: contar celdas de filamento por columna, no asumir
  `w = 2*grosor + 1` (eso es solo el máximo posible de la máscara).
- La R fría "real" de una simulación se lee en el límite `T -> T_0`
  (justificado en el plan §4: al final de SP_set toda celda de filamento está
  exactamente a `T_0`), aplicando `CurrentSolver.mapa_resistencias` sobre la
  matriz de estado **limpia** de percolación (`Clean_state_matrix` — sin
  limpiar, arrastra vacantes aisladas que no forman parte del camino
  conductor y la R sale distinta). Como cross-check se repite el mismo
  procedimiento de ajuste de 0.1 (ventana fría, ajuste por el origen) sobre
  las columnas `V[V]` / `I_total[A]` de `datos_sim`.

In [5]:
def leer_metadata_simulacion(dir_sim, n):
    with open(Path(dir_sim) / f"sim_metadata_{n}.json") as f:
        return json.load(f)


def anchos_reales_por_filamento(cf_matrix_limpia, cf_ranges):
    """Cuenta celdas de filamento por columna dentro de cada banda CF_ranges.

    Devuelve, por filamento, (w_medio, w_min, w_max) sobre el perfil de anchura
    columna a columna. w_min == w_max == w_medio indica filamento rectangular
    regular (la formula cerrada del plan \u00a72.3 seria exacta); si difieren, el
    filamento es irregular y solo vale leer R de la simulacion, no reconstruirla
    (\u00a79, primera trampa).
    """
    perfiles = []
    for fila_min, fila_max in cf_ranges:
        columnas = cf_matrix_limpia[fila_min : fila_max + 1, :].sum(axis=0)
        perfiles.append((float(columnas.mean()), int(columnas.min()), int(columnas.max())))
    return perfiles


def extraer_resultado_simulacion(dir_resultados, n, v_frio=0.25):
    """Extrae (sigma_0_usada, w_total, R_frio_sim) de una simulacion ya ejecutada.

    `dir_resultados` es la carpeta que contiene `simulation_{n}/`.
    """
    dir_sim = Path(dir_resultados) / f"simulation_{n}"
    metadata = leer_metadata_simulacion(dir_sim, n)

    ctes = metadata["ctes_dict"]
    params = metadata["params_dict"]
    cf_ranges = [tuple(r) for r in metadata["cf_ranges"]]

    sigma_0 = ctes["sigma_0"]
    alpha_T = ctes["alpha_T"]
    T_0 = params["init_temp"]
    atom_size = params["atom_size"]
    device_size_x = params["device_size_x"]

    # --- R fria "real": limite T -> T_0, sobre la matriz de estado limpia ---
    estado_final = np.load(dir_sim / f"Final_state_sp_set_{n}.npz")["arr_0"]
    estado_limpio, _ = CurrentSolver.Clean_state_matrix(estado_final)

    R_local = CurrentSolver.mapa_resistencias(estado_limpio, T_0, sigma_0, alpha_T, T_0, atom_size)
    R_por_filamento = CurrentSolver.calcular_resistencia_por_filamento(R_local, cf_ranges)
    R_frio_analitica = CurrentSolver.calcular_resistencia_paralelo(R_por_filamento)

    perfiles_ancho = anchos_reales_por_filamento(estado_limpio, cf_ranges)
    w_total = sum(p[0] for p in perfiles_ancho)

    # --- Cross-check: mismo procedimiento de ajuste que en 0.1, sobre datos_sim ---
    datos_sim = np.load(dir_sim / f"Data_sp_set_{n}.npz")["datos_sim"]
    V, I_total = datos_sim[:, 1], datos_sim[:, 2]  # col 0 NO es el paso, ver \u00a79
    mask_frio = (V > 0) & (V <= v_frio)
    R_frio_fit, _, n_puntos_fit = ajuste_por_origen(V, I_total, mask_frio)

    return {
        "simulacion": n,
        "sigma_0_usada": sigma_0,
        "grosor_filamento": ctes["grosor_filamento"],
        "cf_ranges": cf_ranges,
        "w_total": w_total,
        "perfiles_ancho": perfiles_ancho,
        "R_frio_sim": R_frio_analitica,
        "R_frio_fit_cross_check": R_frio_fit,
        "n_puntos_fit": n_puntos_fit,
        "device_size_x": device_size_x,
        "atom_size": atom_size,
    }

### Nota sobre `Clean_state_matrix`

Sin este paso `R_frio_sim` **no** reproduce la tabla del plan (se comprobó al
escribir este notebook: usar `Final_state_sp_set_{n}.npz` directamente, sin
limpiar, da valores varios % por debajo — arrastra vacantes que no pertenecen
al camino percolante). `Clean_state_matrix` es el mismo filtro que usa el
propio código de simulación antes de resolver corriente (`Clasificar_CF` /
rama eléctrica), así que aplicarlo aquí no es una elección arbitraria del
script de extracción: es replicar el mismo pipeline que ya usa `RRAM/`.

Con la matriz limpia, en `Results_tamano_espacio/simulation_{1,2}` el
filamento sale perfectamente rectangular (`w_min == w_max`), así que la
fórmula cerrada del plan (§2.3, paso C) coincide con `R_frio_sim` de forma
exacta; en `simulation_3` hay una columna con una celda extra (`w_max = w_min
+ 1` en un filamento) — irregularidad pequeña, coherente con la advertencia
del plan de no asumir rectangularidad perfecta en general.

## Uso en la Fase 2

Con la Fase 0 validada, para cada fila de la tabla de barrido (§8, Fase 2) del
plan, una vez lanzada la simulación:

```python
fila = extraer_resultado_simulacion("Results", N)
sigma_0_necesaria = fila["sigma_0_usada"] * (fila["R_frio_sim"] / 41.14)
```

y comparar `sigma_0_necesaria` contra `sigma_0_usada` para decidir si hace
falta iterar (§8, Fase 2, paso 3 — "circularidad de segundo orden").